In [1]:
import geopandas as gpd
import shapely

print("Geopandas:", gpd.__version__)
print("shapely:", shapely.__version__)

Geopandas: 1.1.4
shapely: 2.1.2


In [11]:
import geopandas as gpd
from shapely.geometry import Polygon

polygon1 = Polygon([
    (76.3300, 10.0150),
    (76.3450, 10.0150),
    (76.3450, 10.0300),
    (76.3300, 10.0300),
    (76.3300, 10.0150)
])

polygon2 = Polygon([
    (76.3400, 10.0200),
    (76.3550, 10.0200),
    (76.3550, 10.0350),
    (76.3400, 10.0350),
    (76.3400, 10.0200)
])

In [12]:
gdf1 = gpd.GeoDataFrame(
    {"name": ["Area A"]},
    geometry=[polygon1],
    crs="EPSG:4326"
)

gdf1.to_file(
    "../data/sample_polygon_1.geojson",
    driver="GeoJSON"
)

print("First polygon created!")

gdf2 = gpd.GeoDataFrame(
    {"name": ["Area B"]},
    geometry=[polygon2],
    crs="EPSG:4326"
)

gdf2.to_file(
    "../data/sample_polygon_2.geojson",
    driver="GeoJSON"
)

print("Second polygon created!")

First polygon created!
Second polygon created!


In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [5]:
import os
import geopandas as gpd
import folium

from IPython.display import display, Markdown
from dotenv import load_dotenv
from google import genai
from google.genai import types

from src.gis_tools import (
    calculate_area,
    calculate_distance,
    create_buffer,
    intersect_layers,
)

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY was not found. Check your .env file.")

client = genai.Client(api_key=api_key)
print("✅ Gemini connected")

✅ Gemini connected


## 2. The GIS tools

These small wrapper functions tell Gemini what each GIS operation does. They also save the latest result so we can show it on a map.

In [ ]:
MAP_VISUALIZATION = {"type": None, "data": None}


def calculate_area_tool(file_path: str) -> str:
    """Calculate polygon area in square kilometres."""
    global MAP_VISUALIZATION

    gdf = gpd.read_file(file_path)
    area = calculate_area(file_path)

    MAP_VISUALIZATION = {
        "type": "area",
        "data": {"gdf": gdf, "area": area}
    }

    return f"The polygon area is {area:.2f} square kilometres."


def calculate_distance_tool(
    longitude1: float,
    latitude1: float,
    longitude2: float,
    latitude2: float
) -> str:
    """Calculate distance between two locations in kilometres."""

    distance = calculate_distance(
        (longitude1, latitude1),
        (longitude2, latitude2)
    )

    return f"The distance is {distance:.2f} kilometres."


def create_buffer_tool(file_path: str, distance_meters: float) -> str:
    """Create a buffer around a GIS layer."""
    global MAP_VISUALIZATION

    original = gpd.read_file(file_path)
    buffered = create_buffer(file_path, distance_meters)

    MAP_VISUALIZATION = {
        "type": "buffer",
        "data": {
            "original": original,
            "buffered": buffered,
            "distance": distance_meters
        }
    }

    return f"Created a {distance_meters:g} meter buffer."


def intersect_layers_tool(
    layer1_path: str,
    layer2_path: str
) -> str:
    """Find the overlap between two GIS layers."""
    global MAP_VISUALIZATION

    layer1 = gpd.read_file(layer1_path)
    result = intersect_layers(layer1_path, layer2_path)

    MAP_VISUALIZATION = {
        "type": "intersection",
        "data": {
            "layer1": layer1,
            "layer2": gpd.read_file(layer2_path),
            "intersection": result
        }
    }

    return f"The layers have {len(result)} intersecting feature(s)."

✅ GIS tools ready: area, distance, buffer, intersection


## 3. Give the tools to Gemini

In [7]:
chat = client.chats.create(
    model="gemini-3.6-flash",
    config=types.GenerateContentConfig(
        tools=[
            calculate_area_tool,
            calculate_distance_tool,
            create_buffer_tool,
            intersect_layers_tool,
        ]
    ),
)

print("🤖 AI GIS assistant ready")

🤖 AI GIS assistant ready


## 4. Show the GIS result on a map

After Gemini finishes the tool call, this function shows the map produced by that GIS operation.

In [ ]:
def to_wgs84(gdf):
    if gdf.crs is None:
        return gdf.set_crs(4326, allow_override=True)
    return gdf.to_crs(4326)


def map_center(*gdfs):
    layers = [to_wgs84(g) for g in gdfs if g is not None and not g.empty]
    bounds = [g.total_bounds for g in layers]
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)
    return [(miny + maxy) / 2, (minx + maxx) / 2]


def add_layer(m, gdf, name, fill, line, opacity):
    if gdf.empty:
        return
    folium.GeoJson(
        to_wgs84(gdf).__geo_interface__,
        name=name,
        style_function=lambda feature: {
            "fillColor": fill,
            "color": line,
            "weight": 2,
            "fillOpacity": opacity,
        },
    ).add_to(m)


def show_last_result():
    result = MAP_VISUALIZATION
    if result["type"] is None:
        print("No visual result available.")
        return

    kind = result["type"]
    data = result["data"]

    if kind == "area":
        gdf, area = data["gdf"], data["area"]
        m = folium.Map(location=map_center(gdf), zoom_start=13)
        add_layer(m, gdf, "Polygon", "#3b82f6", "#1d4ed8", 0.35)
        display(Markdown(f"### 📐 Area: **{area:.2f} km²**"))
        display(m)

    elif kind == "buffer":
        original = data["original"]
        buffered = data["buffered"]
        distance = data["distance"]
        m = folium.Map(location=map_center(original, buffered), zoom_start=13)
        add_layer(m, buffered, f"{distance:g} m Buffer", "#6366f1", "#4338ca", 0.25)
        add_layer(m, original, "Original", "#22c55e", "#166534", 0.70)
        display(Markdown(f"### 🔵 Buffer: **{distance:g} m**"))
        display(m)

    elif kind == "intersection":
        layer1 = data["layer1"]
        layer2 = data["layer2"]
        intersection = data["intersection"]
        m = folium.Map(location=map_center(layer1, layer2, intersection), zoom_start=13)
        add_layer(m, layer1, "Layer 1", "#3b82f6", "#1d4ed8", 0.20)
        add_layer(m, layer2, "Layer 2", "#f59e0b", "#b45309", 0.20)
        add_layer(m, intersection, "Overlap", "#ef4444", "#991b1b", 0.75)
        display(Markdown(f"### 🔀 Intersection: **{len(intersection)} feature(s)**"))
        display(m)

## 5. Ask Gemini

Use natural language. Gemini chooses the appropriate GIS function, Python runs it, and the notebook displays the result.

In [ ]:
def ask_ai(question: str, show_map: bool = True):
    global MAP_VISUALIZATION
    MAP_VISUALIZATION = {"type": None, "data": None}

    print(f"💬 You: {question}\n")
    response = chat.send_message(question)

    print("🤖 Gemini:")
    print(response.text)

    if show_map:
        show_last_result()

    return response

## Demo 1 — Area

**Question:** What is the area of this polygon?

In [14]:
ask_ai("What is the area of this polygon: ../data/sample_polygon_1.geojson?")

💬 You: What is the area of this polygon: ../data/sample_polygon_1.geojson?

🤖 Gemini:
The area of the polygon in `../data/sample_polygon_1.geojson` is **2.83 square kilometres**.


### 📐 Area: **2.83 km²**

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='The area of the polygon in `../data/sample_polygon_1.geojson` is **2.83 square kilometres**.',
            thought_signature=b'\x12\xde\x01\n\xdb\x01\x01\x11M2\x0fv\xe1\x1a\x03\xed\xb3S\xb8\x0e\xfcw\x94PQ}?\xab\x8c\x97\x00\x00I\xf7\x94\x02\xea\xe1\x8aG\xcf)\x02\x11\x89\xeaJ\t\xa6\xcd?H|\x9f?\xfc\x19\x8fJ\xde\xd6x\xc6M\xd8\x9e\xf0\xc3\x94:\xba\x96E\xf8\xb8\xb4ah\x1dU>\xd7\x9e\tX\x8e\xfa\\\x02\xec|\x04B.\xd1\xaf\x9d...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.6-flash',
  response_id='u06matSuKYSFjuMP4bXAsAY',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=27,
    prompt_token_count=2533,
    prompt_tokens_details=[
      ModalityTokenCount(
        mo

## Demo 2 — Buffer

**Question:** Create a 500-meter buffer around this area.

In [15]:
ask_ai("Create a 500-meter buffer around ../data/sample_polygon_1.geojson")

💬 You: Create a 500-meter buffer around ../data/sample_polygon_1.geojson

🤖 Gemini:
A 500-meter buffer has been successfully created around `../data/sample_polygon_1.geojson` (1 feature processed).


### 🔵 Buffer: **500 m**

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='A 500-meter buffer has been successfully created around `../data/sample_polygon_1.geojson` (1 feature processed).',
            thought_signature=b'\x12\xdc\x01\n\xd9\x01\x01\x11M2\x0f\x16\xfd\xfaF3b\xd9\xefV\xb0\xe4\xd0@z4\xaa\x06\x06\xd3S$.\xe1\xcd\x10\x84ES\xd4i\xfe\xec)4\x02N-8h\x96\x8a\xa8>(\n\x9e\x94\xc1I\xc9\xcb\xb3\x87\xf6\x85\x1dP\x14Q\x96\x15\x96\xa3\x8c"\x80\xa6\xc4\x81=\x8f8\x89!kR\xf4\x95-\x17\xe8\xcce\x80\xcd\xd7...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.6-flash',
  response_id='41GmasT8Nfyog8UPg9bfiQE',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=30,
    prompt_token_count=2793,
    prompt_tokens_details=[
      ModalityTokenCoun

## Demo 3 — Intersection

**Question:** Where do these two areas overlap?

In [16]:
ask_ai(
    "Find the intersection between ../data/sample_polygon_1.geojson "
    "and ../data/sample_polygon_2.geojson"
)

💬 You: Find the intersection between ../data/sample_polygon_1.geojson and ../data/sample_polygon_2.geojson

🤖 Gemini:
The intersection between `../data/sample_polygon_1.geojson` and `../data/sample_polygon_2.geojson` has been calculated successfully, resulting in 1 intersecting feature.


### 🔀 Intersection: **1 feature(s)**

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='The intersection between `../data/sample_polygon_1.geojson` and `../data/sample_polygon_2.geojson` has been calculated successfully, resulting in 1 intersecting feature.',
            thought_signature=b'\x12\xd6\x01\n\xd3\x01\x01\x11M2\x0f\xfa\x82\xdb\x03\x87\x12=9ECxH\xab\x0f= \xdb\xae\xcd}\x01\x0bE\xa5LWv5\x98\x06\xa5\x86\xf6X+\x1c${\xcfW\xbf\xd3\x14\x93\t\xa21\xe4\x11Tf\xb9\xe6ev\x8b6O\x8a\x0e\xa3\x9a\xa1\xa8F\xf7e\x03\r\xc0\xdf\x07\xa8\xdb\xe9\x92\xc1;\xe9\x88\xe7\xf0\xe4\xfe\xf0y...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.6-flash',
  response_id='MFKmaqfFCLa6mNMP8snauQo',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=40,
    prompt_token_cou